In [18]:
import pandas as pd
import random
import math
df = pd.read_csv('../lessons/lesson_9/accommodation.csv')

In [19]:
# линейная регрессия руками (без sklearn)

quality_values = df["OverallQual"].tolist()   # признак 1: общее качество
area_values    = df["GrLivArea"].tolist()     # признак 2: жилая площадь
price_values   = df["SalePrice"].tolist()     # цена

# логарифмируем цену (выравниваем хвост)
log_price = []
for price in price_values:
    log_price.append(math.log(price))

def select(values, indices):          # выбрать элементы по списку номеров
    result = []
    for i in indices:
        result.append(values[i])
    return result

def mean(values):
    return sum(values) / len(values)

def std_dev(values):
    average = mean(values)
    total = 0
    for value in values:
        total += (value - average)**2
    return (total / len(values))**0.5

def standardize(values, average, spread):
    result = []
    for value in values:
        result.append((value - average) / spread)
    return result

# делим на train / test ===
random.seed(42)
indices = list(range(len(log_price)))
random.shuffle(indices)
split_point   = int(len(indices) * 0.8)
train_indices = indices[:split_point]
test_indices  = indices[split_point:]

quality_train, quality_test = select(quality_values, train_indices), select(quality_values, test_indices)
area_train,    area_test    = select(area_values,    train_indices), select(area_values,    test_indices)
price_train,   price_test   = select(log_price,      train_indices), select(log_price,      test_indices)

# нормализация (mean и std берём только из train)
quality_mean, quality_std = mean(quality_train), std_dev(quality_train)
area_mean,    area_std    = mean(area_train),    std_dev(area_train)

quality_train_norm = standardize(quality_train, quality_mean, quality_std)
quality_test_norm  = standardize(quality_test,  quality_mean, quality_std)
area_train_norm    = standardize(area_train,    area_mean,    area_std)
area_test_norm     = standardize(area_test,     area_mean,    area_std)

# обучаем градиентным спуском
weight_quality = 0.0
weight_area    = 0.0
bias           = mean(price_train)
learning_rate  = 0.1
num_samples    = len(price_train)

for step in range(2000):
    grad_weight_quality = 0.0
    grad_weight_area    = 0.0
    grad_bias           = 0.0
    for quality, area, actual in zip(quality_train_norm, area_train_norm, price_train):
        prediction = weight_quality * quality + weight_area * area + bias
        error      = prediction - actual
        grad_weight_quality += 2 * error * quality
        grad_weight_area    += 2 * error * area
        grad_bias           += 2 * error
    grad_weight_quality /= num_samples
    grad_weight_area    /= num_samples
    grad_bias           /= num_samples
    weight_quality -= learning_rate * grad_weight_quality
    weight_area    -= learning_rate * grad_weight_area
    bias           -= learning_rate * grad_bias

# оцениваем на тесте
def predict(quality, area):
    return weight_quality * quality + weight_area * area + bias

ss_res = 0
for quality, area, actual in zip(quality_test_norm, area_test_norm, price_test):
    ss_res += (predict(quality, area) - actual)**2

test_mean = mean(price_test)
ss_tot = 0
for actual in price_test:
    ss_tot += (actual - test_mean)**2

r2 = 1 - ss_res / ss_tot

errors = []
for quality, area, actual in zip(quality_test_norm, area_test_norm, price_test):
    predicted_price = math.exp(predict(quality, area))
    actual_price    = math.exp(actual)
    errors.append(abs(predicted_price - actual_price))
mean_error = sum(errors) / len(errors)

print("weight_quality:", round(weight_quality, 3))
print("weight_area:   ", round(weight_area, 3))
print("bias:          ", round(bias, 3))
print("R2 на тесте:   ", round(r2, 3))
print("средняя ошибка:", f"${round(mean_error):,}")

weight_quality: 0.242
weight_area:    0.133
bias:           12.013
R2 на тесте:    0.758
средняя ошибка: $27,404


In [ ]:
# коробочное решение

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

X = df[["OverallQual", "GrLivArea"]]     # два признака
y = np.log(df["SalePrice"])              # логарифм цены

# делим на train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# нормализуем (scaler учится на train, применяется к обоим)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm  = scaler.transform(X_test)

# обучаем — весь градиентный спуск спрятан здесь
model = LinearRegression()
model.fit(X_train_norm, y_train)

# оцениваем
predictions = model.predict(X_test_norm)
r2 = r2_score(y_test, predictions)
mean_error = np.abs(np.exp(predictions) - np.exp(y_test)).mean()

print("веса:", [round(c, 3) for c in model.coef_], " bias:", round(model.intercept_, 3))
print("R2 на тесте:", round(r2, 3))
print("средняя ошибка:", f"${round(mean_error):,}")

веса: [np.float64(0.24), np.float64(0.129)]  bias: 12.031
R2 на тесте: 0.785
средняя ошибка: $25,054
